# Data Preprocessing — NIH ChestX-Ray14

**Project:** Multi-Label Thoracic Disease Classification using Deep Learning  
**Dataset:** NIH ChestX-Ray14 — 112,120 frontal chest X-ray images  

---

## Notebook Objectives

This notebook prepares all data artifacts required for model training:

1. Load metadata and official train/test split files
2. Build image folder paths across the 12 dataset archives
3. Extract the 14 disease class labels
4. Split data into train / validation / test sets (patient-level)
5. Compute per-class positive weights for imbalanced loss correction
6. Save all essential artifacts to a single `.pth` file for use in training

> **Note:** No model training occurs here. This notebook outputs `data_essentials.pth` which is loaded directly by `03_training.ipynb`.

## 1. Imports & Device Configuration

In [26]:
import pandas as pd
import os
from torch import device, cuda, float32, tensor, save
from sklearn.model_selection import train_test_split
from warnings import filterwarnings

filterwarnings("ignore")
%matplotlib inline

# Use GPU if available, otherwise fall back to CPU
dev = device("cuda" if cuda.is_available() else "cpu")
dev

device(type='cuda')

## 2. Load Metadata & Official Split Files

The NIH dataset provides two official split files:
- `train_val_list.txt` — image names reserved for training and validation
- `test_list.txt` — image names reserved for final evaluation

These splits were constructed at the **patient level**, meaning all scans from a single patient
appear exclusively in either the train/val set or the test set — never both.
This prevents data leakage and ensures the test set reflects true generalization performance.

In [27]:
# Load full metadata CSV — one row per image
df = pd.read_csv(r"D:\Data\Data_Entry_2017.csv")

# Load official patient-level split files
# header=None: these files have no header row — each line is a single image filename
train_val_names = pd.read_csv(r"D:\Data\train_val_list.txt", header=None, names=["Image Index"])
test_names      = pd.read_csv(r"D:\Data\test_list.txt",      header=None, names=["Image Index"])

## 3. Build Image Folder Paths & Disease Classes

The 112,120 images are distributed across 12 archives (`images_001` to `images_012`),
each extracted into a subfolder named `images/`.

We collect all folder paths dynamically so the Custom Dataset can locate any image
regardless of which archive it belongs to.

In [28]:
# Root directory containing all extracted image archives
# Update this path to match your local dataset location
PATH = r"D:\Data"

# Collect paths for all 12 image subfolders
# Each folder follows the pattern: images_00X/images/
IMG_FOLDERS = [
    os.path.join(PATH, x + "\\images")
    for x in os.listdir(PATH)
    if x.startswith("images_")
]

# Total number of disease classes (excluding 'No Finding')
NUM_OF_CLASSES = 14

# Extract the 14 unique disease names from the dataset
# Sorted alphabetically for consistent label ordering across all notebooks
# 'No Finding' is excluded — represented implicitly as an all-zero label vector
ALL_DISEASES = sorted(
    df[df["Finding Labels"] != "No Finding"]["Finding Labels"]
    .str.split("|")
    .explode()
    .unique()
)

ALL_DISEASES

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

## 4. Prepare Split Sets

We convert the split filename lists to Python sets for efficient `O(1)` lookup,
then filter the full metadata DataFrame accordingly.

In [29]:
# Convert to Python sets for fast membership lookup during DataFrame filtering
train_val_names = set(train_val_names["Image Index"].values)
test_names      = set(test_names["Image Index"].values)

In [30]:
# Keep only the two columns needed for training:
# - Image Index   : filename used to locate the image on disk
# - Finding Labels: pipe-separated disease labels (e.g. 'Pneumonia|Effusion')
df_imgs_labels = df[["Image Index", "Finding Labels"]]
df_imgs_labels

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
4,00000003_000.png,Hernia
...,...,...
112115,00030801_001.png,Mass|Pneumonia
112116,00030802_000.png,No Finding
112117,00030803_000.png,No Finding
112118,00030804_000.png,No Finding


## 5. Train / Validation / Test Split

The official split files define which images belong to train+val vs test.
We further split the train+val portion into:
- **80% Training** — used to update model weights each epoch
- **20% Validation** — used to monitor generalization and trigger early stopping

> `stratify` is intentionally omitted — sklearn's stratify only supports
> single-label problems and cannot handle multi-label distributions.

In [31]:
# Filter metadata to official train+val images only
train_val_df = df_imgs_labels[df_imgs_labels["Image Index"].isin(train_val_names)]
train_val_df

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
12,00000004_000.png,Mass|Nodule
...,...,...
112100,00030789_000.png,Infiltration
112106,00030793_000.png,Mass|Nodule
112108,00030795_000.png,Pleural_Thickening
112114,00030801_000.png,No Finding


In [32]:
# Split train+val into 80% train and 20% validation
# random_state=44 ensures reproducible splits across runs
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=44)

# Filter metadata to official test images only
test_df = df_imgs_labels[df_imgs_labels["Image Index"].isin(test_names)]

print("Train      :", train_df.shape[0], "images")
print("Validation :", val_df.shape[0],   "images")
print("Test       :", test_df.shape[0],  "images")
print("Total      :", train_df.shape[0] + val_df.shape[0] + test_df.shape[0])

Train      : 69219 images
Validation : 17305 images
Test       : 25596 images
Total      : 112120


## 6. Class Imbalance — Positive Weights

The NIH dataset suffers from severe class imbalance:
- `No Finding` accounts for ~54% of all images (~60,361 samples)
- `Hernia` has only ~227 samples — a ratio of approximately 265:1

To address this, we use **Weighted Loss (pos_weight)** inside `BCEWithLogitsLoss`.

For each disease, the weight is:

$$pos\_weight_i = \frac{\text{negative samples}}{\text{positive samples}}$$

A higher weight means the model is penalized more when it misses that disease,
forcing equal attention across all 14 classes regardless of their frequency.

**Why not oversampling or SMOTE?**
- No data duplication → faster training
- No risk of overfitting on repeated rare samples
- Mathematically sound for large-scale medical imaging

In [33]:
total = len(df)
pos_weights = []

for disease in ALL_DISEASES:
    # Count images where this disease appears in the label string
    pos = df["Finding Labels"].str.contains(disease).sum()
    # Count images where this disease is absent
    neg = total - pos
    # pos_weight = neg / pos — higher value = rarer disease = stronger penalty
    pos_weights.append(neg / pos)

# Convert to tensor on the target device (GPU/CPU)
pos_weights = tensor(pos_weights, dtype=float32).to(dev)

print("pos_weight per disease (higher = rarer):")
for name, w in zip(ALL_DISEASES, pos_weights):
    print(f"  {name:25s}: {w:.2f}")

pos_weight per disease (higher = rarer):
  Atelectasis              : 8.70
  Cardiomegaly             : 39.39
  Consolidation            : 23.02
  Edema                    : 47.68
  Effusion                 : 7.42
  Emphysema                : 43.56
  Fibrosis                 : 65.50
  Hernia                   : 492.92
  Infiltration             : 4.64
  Mass                     : 18.39
  Nodule                   : 16.71
  Pleural_Thickening       : 32.12
  Pneumonia                : 77.35
  Pneumothorax             : 20.15


## 7. Save All Artifacts

All preprocessing outputs are bundled into a single `data_essentials.pth` file.
This avoids re-running this notebook every time training is started.

**Saved artifacts:**

| Key | Type | Description |
|-----|------|-------------|
| `train_df` | DataFrame | Training images and labels |
| `val_df` | DataFrame | Validation images and labels |
| `test_df` | DataFrame | Test images and labels |
| `all_diseases` | list | Sorted list of 14 disease class names |
| `img_folders` | list | Paths to the 12 image archive folders |
| `num_of_classes` | int | 14 |
| `pos_weights` | Tensor | Per-class weights for BCEWithLogitsLoss |

In [34]:
save({
    'train_df'       : train_df,
    'val_df'         : val_df,
    'test_df'        : test_df,
    'all_diseases'   : ALL_DISEASES,
    'img_folders'    : IMG_FOLDERS,
    'num_of_classes' : NUM_OF_CLASSES,
    'pos_weights'    : pos_weights,
}, 'data_essentials.pth')

print("Saved: data_essentials.pth")
print("Load in 03_training.ipynb using: torch.load('data_essentials.pth')")

Saved: data_essentials.pth
Load in 03_training.ipynb using: torch.load('data_essentials.pth')
